# Kernels in the Machine
## How Kernel Swapping Affects Model Performance

---

Modern LLM inference stacks are increasingly modular. HuggingFace's [kernel hub](https://huggingface.co/kernels-community) lets practitioners drop in optimized, third-party CUDA or Triton kernels for individual operations — RMSNorm, attention, MLP — without touching model weights or architecture.

The appeal is obvious: faster kernels, same model. But this modularity hides a subtle trap.

> *If a kernel passes its correctness tests within atol/rtol tolerance, does that guarantee the model behaves identically?*

**No** — and this post shows exactly why.

The core issue is **error accumulation**. A Transformer is a deep composition of operations across many layers. A tiny floating-point deviation introduced by a swapped kernel in layer 1 doesn't stay tiny — it propagates forward, gets scaled, added to residual streams, normalized again, and amplified through attention and feed-forward projections. By the final layer, the model may be in a meaningfully different hidden state, generating different tokens and scoring differently on downstream benchmarks.

We'll demonstrate this using [`Qwen/Qwen3.5-0.8B`](https://huggingface.co/Qwen/Qwen3.5-0.8B) with the [`kernels-community/tinygrad-rms`](https://huggingface.co/kernels-community/tinygrad-rms) kernel, comparing behavior across both `bf16` and `fp16` precision.

## Setup

In [1]:
%%capture
!pip install torch transformers kernels datasets matplotlib seaborn numpy tqdm accelerate

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import string
import gc
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from kernels import get_kernel

MODEL_NAME = "Qwen/Qwen3.5-0.8B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_TRIVIA_SAMPLES = 300

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---

## RMSNorm: The Operation Under the Microscope

Root Mean Square Normalization ([Zhang & Sennrich, 2019](https://arxiv.org/abs/1910.07467)) is the normalization layer of choice in most modern LLMs — Qwen, LLaMA, Mistral all use it. Its formula is deceptively simple:

$$\text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x})} \cdot \mathbf{w}, \quad \text{where} \quad \text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

Unlike LayerNorm, RMSNorm skips the mean-subtraction step, making it cheaper to compute. But that simplicity is exactly what makes it a good optimization target — and a good case study for kernel swapping effects.

Here's a reference PyTorch implementation:

In [ ]:
class ReferenceRMSNorm(nn.Module):
    """Numerically explicit RMSNorm for comparison."""
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        input_dtype = hidden_states.dtype
        x = hidden_states.float()
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.variance_epsilon)
        return (self.weight * x).to(input_dtype)

---

## Dropping In a Custom Kernel

HuggingFace's `kernels` package makes it straightforward to load community-contributed implementations of standard operations. The `kernels-community/tinygrad-rms` kernel is a TinyGrad-compiled RMSNorm — designed to be a fast, drop-in replacement.

Let's load it and inspect what it exposes:

In [ ]:
rms_kernel = get_kernel("kernels-community/tinygrad-rms", version=1)

print("Kernel type  :", type(rms_kernel))
print("Public attrs :", [a for a in dir(rms_kernel) if not a.startswith("_")])

In [ ]:
def call_rms_kernel(
    kernel, hidden_states: torch.Tensor, weight: torch.Tensor, eps: float
) -> torch.Tensor:
    """Adaptively invoke the kernel using whichever calling convention it exposes."""
    for method_name in ["rms_norm", "forward"]:
        fn = getattr(kernel, method_name, None)
        if callable(fn):
            try:
                return fn(hidden_states, weight, eps)
            except TypeError:
                pass
    try:
        return kernel(hidden_states, weight, eps)
    except TypeError:
        pass
    raise RuntimeError(
        f"No working calling convention found. Available: {[a for a in dir(kernel) if not a.startswith('_')]}"
    )

---

## Part 1: Correctness Tests — And They Pass

The standard way to validate a custom kernel is to compare its outputs against a reference implementation on random inputs, within some numerical tolerance. This is the test that would appear in a kernel's CI suite.

We'll test across `float32`, `bfloat16`, and `float16`, using `atol=1e-3, rtol=1e-3` — a reasonable tolerance for mixed-precision arithmetic:

In [ ]:
def test_kernel_correctness(
    kernel, dtype: torch.dtype, hidden_size: int = 1024,
    atol: float = 1e-3, rtol: float = 1e-3, seed: int = 42
):
    torch.manual_seed(seed)
    batch, seq_len = 2, 16

    hidden_states = torch.randn(batch, seq_len, hidden_size, dtype=dtype, device=DEVICE)
    weight = torch.ones(hidden_size, dtype=dtype, device=DEVICE)
    eps = 1e-6

    ref = ReferenceRMSNorm(hidden_size, eps=eps).to(dtype).to(DEVICE)
    ref.weight.data = weight.clone()

    with torch.no_grad():
        ref_out = ref(hidden_states)
        kernel_out = call_rms_kernel(kernel, hidden_states, weight, eps)

    passed = torch.allclose(ref_out.float(), kernel_out.float(), atol=atol, rtol=rtol)
    max_diff = (ref_out.float() - kernel_out.float()).abs().max().item()
    mean_diff = (ref_out.float() - kernel_out.float()).abs().mean().item()
    return passed, max_diff, mean_diff


print(f"{'dtype':<12} {'passed':<10} {'max |diff|':<15} {'mean |diff|':<15}")
print("-" * 52)
for dtype_name, dtype in [
    ("float32",   torch.float32),
    ("bfloat16",  torch.bfloat16),
    ("float16",   torch.float16),
]:
    passed, max_diff, mean_diff = test_kernel_correctness(rms_kernel, dtype)
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{dtype_name:<12} {status:<10} {max_diff:<15.2e} {mean_diff:<15.2e}")

The kernel passes. Small differences exist (floating-point arithmetic is not commutative), but they're well within tolerance. By any standard kernel validation criterion, this is a green light.

Now let's see what happens when we plug it into an actual model.

---

## Part 2: Into the Model — The Accumulation Problem

A Transformer decoder like Qwen3.5-0.8B applies RMSNorm at multiple points per layer: before self-attention, after self-attention (in some architectures), and before the MLP. With 28 layers, that's on the order of **56+ RMSNorm calls** per forward pass.

Each call introduces a small perturbation $\delta_i$. The residual stream at layer $l$ is roughly:

$$\mathbf{h}_l = \mathbf{h}_{l-1} + f_l(\text{RMSNorm}(\mathbf{h}_{l-1}))$$

So the perturbation from layer 1 gets added into $\mathbf{h}_1$, which is then passed through $f_2$, which amplifies or reshapes it, and so on. The perturbations don't cancel — they compound.

Let's load the model and build the patching infrastructure to test this.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_NAME} in bfloat16...")
model_bf16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
)
model_bf16.eval()

print(f"Hidden size : {model_bf16.config.hidden_size}")
print(f"Num layers  : {model_bf16.config.num_hidden_layers}")
print(f"Parameters  : {sum(p.numel() for p in model_bf16.parameters()) / 1e6:.0f}M")

In [ ]:
def _get_rmsnorm_params(module):
    weight = module.weight
    eps = getattr(module, "variance_epsilon", getattr(module, "eps", 1e-6))
    return weight, eps


def patch_model_rmsnorm(model, kernel):
    """Replace every RMSNorm layer's forward with the kernel implementation."""
    original_forwards = {}
    for name, module in model.named_modules():
        if "RMSNorm" in type(module).__name__:
            weight, eps = _get_rmsnorm_params(module)
            original_forwards[name] = module.forward

            def _make_forward(w, e):
                def _forward(hidden_states):
                    return call_rms_kernel(kernel, hidden_states, w, e)
                return _forward

            module.forward = _make_forward(weight, eps)
    print(f"Patched {len(original_forwards)} RMSNorm modules")
    return original_forwards


def unpatch_model_rmsnorm(model, original_forwards):
    """Restore all RMSNorm layers to their original forward methods."""
    for name, module in model.named_modules():
        if name in original_forwards:
            module.forward = original_forwards[name]
    print(f"Unpatched {len(original_forwards)} RMSNorm modules")

---

## Part 3: Where Outputs First Diverge

With greedy decoding (`do_sample=False`), two identical models must produce identical tokens. Any difference in the internal computation will eventually tip the argmax at some token position — once the paths diverge, they stay diverged.

Let's run the same prompts through both the original model and the kernel-swapped model and find where they first disagree:

In [ ]:
TEST_PROMPTS = [
    "The capital of France is",
    "In 1969, NASA's Apollo 11 mission successfully landed humans on",
    "The chemical symbol for gold is",
]
MAX_NEW_TOKENS = 40


def greedy_generate(model, tokenizer, prompt: str, max_new_tokens: int = 40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][input_len:].cpu().tolist()
    text = tokenizer.decode(new_ids, skip_special_tokens=True)
    return text, new_ids


divergence_data = []

for prompt in TEST_PROMPTS:
    text_orig, ids_orig = greedy_generate(model_bf16, tokenizer, prompt, MAX_NEW_TOKENS)

    orig_fwd = patch_model_rmsnorm(model_bf16, rms_kernel)
    text_swap, ids_swap = greedy_generate(model_bf16, tokenizer, prompt, MAX_NEW_TOKENS)
    unpatch_model_rmsnorm(model_bf16, orig_fwd)

    divergence_point = next(
        (i for i, (a, b) in enumerate(zip(ids_orig, ids_swap)) if a != b), None
    )
    divergence_data.append({
        "prompt": prompt,
        "original": text_orig,
        "swapped": text_swap,
        "ids_orig": ids_orig,
        "ids_swap": ids_swap,
        "divergence_point": divergence_point,
    })

for d in divergence_data:
    print(f"Prompt   : {d['prompt']!r}")
    print(f"Original : {d['original']!r}")
    print(f"Swapped  : {d['swapped']!r}")
    dp = d['divergence_point']
    print(f"Diverges at token position: {dp if dp is not None else 'identical'}")
    print()

In [ ]:
fig, axes = plt.subplots(len(divergence_data), 1, figsize=(14, 3.5 * len(divergence_data)))
if len(divergence_data) == 1:
    axes = [axes]

for ax, d in zip(axes, divergence_data):
    ids_o = d["ids_orig"]
    ids_s = d["ids_swap"]
    n = min(len(ids_o), len(ids_s), MAX_NEW_TOKENS)

    match = [1 if ids_o[i] == ids_s[i] else 0 for i in range(n)]
    colors = ["#4CAF50" if m else "#F44336" for m in match]

    orig_tokens  = [tokenizer.decode([t]) for t in ids_o[:n]]
    swapped_tokens = [tokenizer.decode([t]) for t in ids_s[:n]]

    for i, (color, ot, st) in enumerate(zip(colors, orig_tokens, swapped_tokens)):
        ax.bar(i, 1, color=color, alpha=0.7, edgecolor="white", linewidth=0.5)
        ax.text(i, 1.05, repr(ot)[:6], ha="center", va="bottom", fontsize=6, color="#1565C0",
                rotation=45)
        ax.text(i, -0.15, repr(st)[:6], ha="center", va="top", fontsize=6, color="#B71C1C",
                rotation=45)

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.4, 1.6)
    ax.set_yticks([])
    ax.set_title(f"Prompt: {d['prompt']!r}", fontsize=10, fontweight="bold")
    ax.set_xlabel("Token position (blue=original, red=swapped)", fontsize=9)
    dp = d["divergence_point"]
    label = f"First divergence at position {dp}" if dp is not None else "No divergence"
    ax.legend(
        handles=[
            mpatches.Patch(color="#4CAF50", label="Tokens match"),
            mpatches.Patch(color="#F44336", label="Tokens differ"),
        ],
        title=label, loc="upper right", fontsize=8,
    )

plt.suptitle(
    "Greedy Decoding: Token Divergence Between Original and Kernel-Swapped Model (bf16)",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig("greedy_divergence_bf16.png", dpi=150, bbox_inches="tight")
plt.show()

---

## Part 4: Layer-by-Layer Error Accumulation

The divergence in output tokens is a symptom. Let's look at the underlying cause: how hidden states drift apart across the depth of the network.

We'll attach forward hooks to every decoder layer to capture the hidden state tensor after each layer's residual addition. Then we compute the L2 norm of the difference between original and swapped at each `(layer, token_position)` point — giving us a 2D heatmap of where and how much the representations diverge.

We run this for both **bf16** and **fp16** to see whether precision affects the accumulation pattern.

In [ ]:
class HiddenStateCapture:
    """Registers forward hooks on decoder layers to capture hidden states."""

    def __init__(self):
        self.states: dict[int, torch.Tensor] = {}
        self._hooks = []

    def register(self, model):
        for i, layer in enumerate(model.model.layers):
            def _hook(module, inp, out, idx=i):
                hs = out[0] if isinstance(out, tuple) else out
                self.states[idx] = hs.detach().cpu().float()
            self._hooks.append(layer.register_forward_hook(_hook))

    def remove(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def clear(self):
        self.states.clear()


def capture_hidden_states(model, input_ids: torch.Tensor) -> dict[int, torch.Tensor]:
    cap = HiddenStateCapture()
    cap.register(model)
    with torch.no_grad():
        model(input_ids.to(model.device))
    cap.remove()
    return cap.states


def compute_diff_matrix(
    states_orig: dict, states_swap: dict
) -> np.ndarray:
    """Returns (n_layers, seq_len) matrix of per-token L2 norms."""
    n_layers = len(states_orig)
    seq_len  = states_orig[0].shape[1]
    mat = np.zeros((n_layers, seq_len))
    for l in range(n_layers):
        diff = states_orig[l][0] - states_swap[l][0]   # [seq_len, hidden_dim]
        mat[l] = diff.norm(dim=-1).numpy()
    return mat


def plot_hidden_state_heatmap(
    diff_matrix: np.ndarray,
    title: str,
    token_labels: list[str] | None = None,
    ax=None,
    vmax=None,
):
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        diff_matrix, aspect="auto", cmap="magma", origin="lower",
        vmin=0, vmax=vmax or diff_matrix.max(),
    )
    plt.colorbar(im, ax=ax, label="L2 Norm of Difference")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Token Position", fontsize=10)
    ax.set_ylabel("Layer Index", fontsize=10)
    if token_labels and len(token_labels) <= 40:
        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=45, ha="right", fontsize=7)
    if standalone:
        plt.tight_layout()
        plt.savefig(f"hidden_diff_{title.lower().replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
ANALYSIS_PROMPT = "In 1969, NASA's Apollo 11 mission successfully landed the first humans on the Moon."
analysis_input_ids = tokenizer(ANALYSIS_PROMPT, return_tensors="pt")["input_ids"]
token_labels = [tokenizer.decode([t]) for t in analysis_input_ids[0].tolist()]

print(f"Analysing {len(token_labels)} tokens in bf16...")

states_bf16_orig = capture_hidden_states(model_bf16, analysis_input_ids)

orig_fwd = patch_model_rmsnorm(model_bf16, rms_kernel)
states_bf16_swap = capture_hidden_states(model_bf16, analysis_input_ids)
unpatch_model_rmsnorm(model_bf16, orig_fwd)

diff_bf16 = compute_diff_matrix(states_bf16_orig, states_bf16_swap)
print(f"bf16 max L2 diff across all layers/positions: {diff_bf16.max():.4f}")
print(f"bf16 mean L2 diff: {diff_bf16.mean():.4f}")

In [ ]:
plot_hidden_state_heatmap(
    diff_bf16,
    title="Hidden State L2 Difference — Original vs Swapped Kernel (bf16)",
    token_labels=token_labels,
)

In [ ]:
del model_bf16
gc.collect()
torch.cuda.empty_cache()

print(f"Loading {MODEL_NAME} in float16...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map=DEVICE,
)
model_fp16.eval()
print("Done.")

In [ ]:
print(f"Analysing {len(token_labels)} tokens in fp16...")

states_fp16_orig = capture_hidden_states(model_fp16, analysis_input_ids)

orig_fwd = patch_model_rmsnorm(model_fp16, rms_kernel)
states_fp16_swap = capture_hidden_states(model_fp16, analysis_input_ids)
unpatch_model_rmsnorm(model_fp16, orig_fwd)

diff_fp16 = compute_diff_matrix(states_fp16_orig, states_fp16_swap)
print(f"fp16 max L2 diff across all layers/positions: {diff_fp16.max():.4f}")
print(f"fp16 mean L2 diff: {diff_fp16.mean():.4f}")

In [ ]:
plot_hidden_state_heatmap(
    diff_fp16,
    title="Hidden State L2 Difference — Original vs Swapped Kernel (fp16)",
    token_labels=token_labels,
)

In [ ]:
shared_vmax = max(diff_bf16.max(), diff_fp16.max())

fig, axes = plt.subplots(1, 2, figsize=(22, 7), sharey=True)
plot_hidden_state_heatmap(
    diff_bf16, "bf16", token_labels=token_labels, ax=axes[0], vmax=shared_vmax
)
plot_hidden_state_heatmap(
    diff_fp16, "fp16", token_labels=token_labels, ax=axes[1], vmax=shared_vmax
)

fig.suptitle(
    "Hidden State Drift: Original vs Swapped RMSNorm Kernel (shared scale)",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig("hidden_diff_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nbf16 mean drift : {diff_bf16.mean():.4f}")
print(f"fp16 mean drift : {diff_fp16.mean():.4f}")
print(f"fp16/bf16 ratio : {diff_fp16.mean() / diff_bf16.mean():.2f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

layer_mean_bf16 = diff_bf16.mean(axis=1)
layer_mean_fp16 = diff_fp16.mean(axis=1)
layers = np.arange(len(layer_mean_bf16))

ax.plot(layers, layer_mean_bf16, label="bf16", color="#1976D2", linewidth=2, marker="o", markersize=4)
ax.plot(layers, layer_mean_fp16, label="fp16", color="#D32F2F", linewidth=2, marker="s", markersize=4)

ax.fill_between(layers, layer_mean_bf16, alpha=0.15, color="#1976D2")
ax.fill_between(layers, layer_mean_fp16, alpha=0.15, color="#D32F2F")

ax.set_xlabel("Layer Index", fontsize=12)
ax.set_ylabel("Mean L2 Norm of Hidden State Difference", fontsize=12)
ax.set_title(
    "Error Accumulation by Layer: Kernel-Swapped vs Original\n"
    "(mean across all token positions)",
    fontsize=13, fontweight="bold",
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("layer_drift_by_precision.png", dpi=150, bbox_inches="tight")
plt.show()

The layer-mean plot tells the story clearly: error does not stay constant across depth. It accumulates — sometimes monotonically, sometimes with spikes at layers that contain more normalization steps (e.g., pre/post-attention norms).

Notice also the difference between `fp16` and `bf16`. Float16 has 10 mantissa bits (higher precision) but a narrower exponent range; bfloat16 has only 7 mantissa bits but the same exponent range as float32. The relative magnitudes of their divergence curves reflect these tradeoffs in how each format rounds intermediate results.

---

## Part 5: Does It Matter for Real Tasks? TriviaQA Benchmark

Hidden state drift is interesting, but the practical question is: **does it change benchmark performance?**

We use [TriviaQA](https://huggingface.co/datasets/trivia_qa) (`rc.nocontext` split), a factual question-answering benchmark. It's well-suited for this experiment because:

- Answers are factual and knowledge-based (not creative), so exact-match scoring is meaningful
- Questions are short, so the model's answer depends heavily on how well it recalls facts — making it sensitive to subtle shifts in the final hidden state
- It's a standard benchmark with established baselines

We evaluate all four configurations:
- **Original kernel, bf16**
- **Swapped kernel, bf16**
- **Original kernel, fp16**
- **Swapped kernel, fp16**

In [ ]:
dataset = load_dataset("trivia_qa", "rc.nocontext", split="validation", trust_remote_code=True)
trivia_samples = dataset.select(range(N_TRIVIA_SAMPLES))
print(f"Loaded {len(trivia_samples)} TriviaQA samples")
print("\nSample entry:")
sample = trivia_samples[0]
print(f"  Question : {sample['question']}")
print(f"  Answer   : {sample['answer']['value']}")
print(f"  Aliases  : {sample['answer']['aliases'][:3]} ...")

In [ ]:
def normalize_answer(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    return " ".join(s.split())


def exact_match(prediction: str, ground_truths: list[str]) -> bool:
    norm_pred = normalize_answer(prediction)
    return any(normalize_answer(gt) == norm_pred for gt in ground_truths)


def evaluate_on_triviaqa(
    model, tokenizer, dataset, n_samples: int, config_name: str = ""
) -> tuple[float, list]:
    correct = 0
    records = []

    for sample in tqdm(dataset.select(range(n_samples)), desc=f"Eval {config_name}"):
        question = sample["question"]
        ground_truths = sample["answer"]["aliases"] + [sample["answer"]["value"]]

        prompt = f"Question: {question}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        prediction = tokenizer.decode(
            out[0][input_len:], skip_special_tokens=True
        ).strip().split("\n")[0].strip()

        is_correct = exact_match(prediction, ground_truths)
        if is_correct:
            correct += 1
        records.append({"question": question, "prediction": prediction, "correct": is_correct})

    return correct / n_samples, records

In [ ]:
print("=== fp16 benchmark ===")

acc_fp16_orig, records_fp16_orig = evaluate_on_triviaqa(
    model_fp16, tokenizer, trivia_samples, N_TRIVIA_SAMPLES, "fp16-original"
)
print(f"fp16 original  : {acc_fp16_orig*100:.2f}%")

orig_fwd = patch_model_rmsnorm(model_fp16, rms_kernel)
acc_fp16_swap, records_fp16_swap = evaluate_on_triviaqa(
    model_fp16, tokenizer, trivia_samples, N_TRIVIA_SAMPLES, "fp16-swapped"
)
unpatch_model_rmsnorm(model_fp16, orig_fwd)
print(f"fp16 swapped   : {acc_fp16_swap*100:.2f}%")

In [ ]:
del model_fp16
gc.collect()
torch.cuda.empty_cache()

print(f"Re-loading {MODEL_NAME} in bfloat16 for benchmark...")
model_bf16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
)
model_bf16.eval()
print("Done.")

In [ ]:
print("=== bf16 benchmark ===")

acc_bf16_orig, records_bf16_orig = evaluate_on_triviaqa(
    model_bf16, tokenizer, trivia_samples, N_TRIVIA_SAMPLES, "bf16-original"
)
print(f"bf16 original  : {acc_bf16_orig*100:.2f}%")

orig_fwd = patch_model_rmsnorm(model_bf16, rms_kernel)
acc_bf16_swap, records_bf16_swap = evaluate_on_triviaqa(
    model_bf16, tokenizer, trivia_samples, N_TRIVIA_SAMPLES, "bf16-swapped"
)
unpatch_model_rmsnorm(model_bf16, orig_fwd)
print(f"bf16 swapped   : {acc_bf16_swap*100:.2f}%")

In [ ]:
results = {
    "Original\n(bf16)": acc_bf16_orig,
    "Swapped\n(bf16)": acc_bf16_swap,
    "Original\n(fp16)": acc_fp16_orig,
    "Swapped\n(fp16)": acc_fp16_swap,
}

fig, ax = plt.subplots(figsize=(10, 6))

configs  = list(results.keys())
accs     = [v * 100 for v in results.values()]
colors   = ["#1565C0", "#EF6C00", "#1565C0", "#EF6C00"]
hatches  = ["", "", "//", "//"]

bars = ax.bar(
    configs, accs, color=colors, hatch=hatches,
    edgecolor="black", linewidth=1.2, width=0.55,
)

for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=11,
    )

y_lo = max(0, min(accs) - 5)
ax.set_ylim(y_lo, max(accs) + 5)
ax.set_ylabel("Exact Match Accuracy (%)", fontsize=12)
ax.set_title(
    f"TriviaQA Exact Match — Original vs Swapped RMSNorm Kernel\n"
    f"({N_TRIVIA_SAMPLES} validation samples, Qwen3.5-0.8B)",
    fontsize=13, fontweight="bold",
)
ax.grid(axis="y", alpha=0.3)

legend_elements = [
    mpatches.Patch(facecolor="#1565C0", edgecolor="black", label="Original kernel"),
    mpatches.Patch(facecolor="#EF6C00", edgecolor="black", label="Swapped kernel"),
    mpatches.Patch(facecolor="white",   edgecolor="black", label="bf16 (solid fill)"),
    mpatches.Patch(facecolor="white",   edgecolor="black", hatch="//", label="fp16 (hatched)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

plt.tight_layout()
plt.savefig("trivia_qa_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDelta (original - swapped):")
print(f"  bf16: {(acc_bf16_orig - acc_bf16_swap)*100:+.2f} pp")
print(f"  fp16: {(acc_fp16_orig - acc_fp16_swap)*100:+.2f} pp")

In [ ]:
orig_correct  = {r["question"] for r in records_bf16_orig if r["correct"]}
swap_correct  = {r["question"] for r in records_bf16_swap if r["correct"]}

orig_only = orig_correct - swap_correct
swap_only = swap_correct - orig_correct

print(f"Questions correct in original ONLY : {len(orig_only)}")
print(f"Questions correct in swapped ONLY  : {len(swap_only)}")

print("\nSample questions where models disagree (original correct, swapped wrong):")
orig_map = {r["question"]: r for r in records_bf16_orig}
swap_map = {r["question"]: r for r in records_bf16_swap}

for q in list(orig_only)[:3]:
    print(f"  Q: {q}")
    print(f"     Original prediction : {orig_map[q]['prediction']!r} ✓")
    print(f"     Swapped  prediction : {swap_map[q]['prediction']!r} ✗")
    print()

---

## Takeaways

### 1. Kernel-level correctness is necessary but not sufficient
The tinygrad RMSNorm kernel passes atol/rtol correctness checks in float32, bf16, and fp16. By every standard unit-test criterion, it is a valid drop-in replacement. Yet the model with the swapped kernel produces different outputs under greedy decoding and scores differently on TriviaQA.

### 2. Errors accumulate — and they accumulate non-linearly
The layer-by-layer hidden state drift plots show that the divergence between original and swapped representations grows with depth, and not uniformly. Layers that apply normalization multiple times (e.g., before and after attention) tend to show sharper jumps.

### 3. Precision matters
fp16 and bf16 differ in how they amplify these errors. bfloat16's reduced mantissa precision (7 bits vs fp16's 10) means individual RMSNorm outputs deviate more — but fp16's narrower exponent range means it can struggle in different regimes. Neither is uniformly better; the right choice depends on the distribution of your activation values.

### 4. The right testing level is model-level, not kernel-level
If you're swapping kernels in a production model, the correct validation protocol is:
1. Pass standard kernel unit tests (atol/rtol) ✓ — necessary
2. Run end-to-end generation comparison with greedy decoding on held-out prompts
3. Evaluate on a task-relevant benchmark with the exact dtype you'll use in production
4. Monitor hidden-state drift across layers as a diagnostic

Steps 2–4 are not standard practice today, but this analysis shows they should be.

---

*Code for this post is available in the companion repository. The `kernels` package used here is [`pip install kernels`](https://pypi.org/project/kernels/) from HuggingFace.*